# 10 — Phân tích lỗi dual-stream top-50

Notebook này **không train**. Nó đọc log, prediction và checkpoint report do notebook 09 lưu trên Drive, sau đó tạo confusion matrix, precision/recall/F1 từng từ, Macro-F1, confidence, các cặp nhầm và báo cáo generalization gap. Nó cũng in trạng thái early stopping và bộ giám sát overfit.

Mặc định chỉ phân tích `validation`. Không chuyển sang `test` để thử cấu hình; chỉ bật test một lần sau khi đã chốt kiến trúc bằng validation.

In [ ]:
from pathlib import Path
from google.colab import drive

PROJECT_GIT_URL = 'https://github.com/stillthethrone/silent-signal.git'
PROJECT_GIT_REF = 'feat/asl-citizen-dual-stream-demo50'  # @param {type:'string'}
RESULTS_ROOT_STR = '/content/drive/.shortcut-targets-by-id/1-oYEcvJh4ylv_f4AKkJkCjs3FgzjDBFE/silent-signal-results/asl_citizen'  # @param {type:'string'}
BASELINE_RUN_NAME = 'videomaev2_rgb_transformer_demo50_e50_compact_v1'  # @param {type:'string'}
DUAL_RUN_NAME = 'videomaev2_graph_spatial_temporal_crossattn_demo50_v2'  # @param {type:'string'}
ANALYSIS_SPLIT = 'validation'  # @param ['validation', 'test']
ALLOW_TEST_ANALYSIS = False  # @param {type:'boolean'}
TOP_ERRORS = 15  # @param {type:'integer'}
RUN_ANALYSIS = True  # @param {type:'boolean'}

DRIVE_MOUNT = Path('/content/drive')
def drive_ready():
    return (DRIVE_MOUNT / 'MyDrive').exists() or (DRIVE_MOUNT / '.shortcut-targets-by-id').exists()
if not drive_ready():
    try:
        drive.mount(str(DRIVE_MOUNT), force_remount=False, timeout_ms=120_000)
    except ValueError as first_error:
        print('Drive mount lần đầu thất bại; thử force_remount một lần...', flush=True)
        try:
            drive.mount(str(DRIVE_MOUNT), force_remount=True, timeout_ms=120_000)
        except ValueError as retry_error:
            raise RuntimeError('Không mount được Drive. Hãy disconnect runtime, kết nối lại đúng tài khoản rồi chạy lại.') from retry_error
if not drive_ready(): raise RuntimeError('Drive mount xong nhưng không thấy MyDrive/shortcut targets.')
if ANALYSIS_SPLIT == 'test' and not ALLOW_TEST_ANALYSIS:
    raise RuntimeError('Không dùng test để chỉnh mô hình. Chỉ bật sau khi đã chốt bằng validation.')
PROJECT_ROOT = Path('/content/silent-signal')
TOP200_ROOT = Path(RESULTS_ROOT_STR) / 'subsets/asl_citizen_asllex_top200'
BASELINE_ROOT = TOP200_ROOT / 'baselines' / BASELINE_RUN_NAME
DUAL_ROOT = TOP200_ROOT / 'experiments' / DUAL_RUN_NAME
SELECTED_WORDS = BASELINE_ROOT / 'selected_50_words.json'
DUAL_REPORT = DUAL_ROOT / 'dual_stream_report.json'
print('Drive READY:', DRIVE_MOUNT, flush=True)

## Môi trường phân tích CPU cô lập và kiểm tra artifact

In [ ]:
import os, subprocess, sys, time
def run(command, *, env=None):
    command = list(map(str, command))
    started = time.perf_counter(); print('+', ' '.join(command), flush=True)
    subprocess.run(command, check=True, env=env)
    print(f'DONE {time.perf_counter() - started:.1f}s', flush=True)
if not PROJECT_ROOT.exists():
    run(['git', 'clone', '--branch', PROJECT_GIT_REF, '--single-branch', PROJECT_GIT_URL, PROJECT_ROOT])
else:
    run(['git', '-C', PROJECT_ROOT, 'fetch', 'origin', PROJECT_GIT_REF])
    run(['git', '-C', PROJECT_ROOT, 'checkout', PROJECT_GIT_REF])
    run(['git', '-C', PROJECT_ROOT, 'pull', '--ff-only', 'origin', PROJECT_GIT_REF])
ANALYSIS_SITE = Path('/content/silent-signal-analysis-site-py313-v1')
ANALYSIS_SITE.mkdir(parents=True, exist_ok=True)
analysis_packages = ['numpy==2.2.2', 'scipy==1.15.1', 'pandas==2.2.3', 'scikit-learn==1.6.1', 'seaborn==0.13.2', 'matplotlib==3.10.0']
import_check = 'import numpy, scipy, pandas, sklearn, seaborn, matplotlib; print("Environment PASS | numpy", numpy.__version__, "| pandas", pandas.__version__, "| sklearn", sklearn.__version__)'
ANALYSIS_ENV = dict(os.environ)
ANALYSIS_ENV['PYTHONPATH'] = os.pathsep.join([str(ANALYSIS_SITE), str(PROJECT_ROOT / 'src')])
probe = subprocess.run([sys.executable, '-c', import_check], text=True, capture_output=True, env=ANALYSIS_ENV)
if probe.returncode != 0:
    print('Cài analysis stack tương thích vào thư mục cô lập...', flush=True)
    run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-cache-dir', '--upgrade', '--target', ANALYSIS_SITE, *analysis_packages])
run([sys.executable, '-c', import_check], env=ANALYSIS_ENV)
required = [SELECTED_WORDS, DUAL_REPORT, DUAL_ROOT / f'{ANALYSIS_SPLIT}_predictions.csv']
missing = [str(path) for path in required if not path.is_file()]
if missing: raise FileNotFoundError('Notebook 09 chưa tạo đủ artifact: ' + ', '.join(missing))
print('Input PASS:', *required, sep='\n- ')

## Chạy phân tích và tạo dashboard

In [ ]:
command = [sys.executable, '-u', '-m', 'silent_signal.cli.analyze_videomaev2_demo',
    '--run-root', DUAL_ROOT, '--selection-report', SELECTED_WORDS,
    '--training-report', DUAL_REPORT,
    '--model-label', 'VideoMAE V2 + Graph-Spatial-Temporal Transformer + Cross-Attention',
    '--split', ANALYSIS_SPLIT, '--top-errors', TOP_ERRORS]
if RUN_ANALYSIS: run(command, env=ANALYSIS_ENV)
else: print('RUN_ANALYSIS=False — chưa tạo báo cáo lỗi.')

## Log dừng, overfit và biểu đồ lỗi

In [ ]:
import json
from IPython.display import Image, display
training = json.loads(DUAL_REPORT.read_text(encoding='utf-8'))
analysis_path = DUAL_ROOT / f'{ANALYSIS_SPLIT}_error_analysis.json'
if not analysis_path.is_file(): raise FileNotFoundError(analysis_path)
analysis = json.loads(analysis_path.read_text(encoding='utf-8'))
summary = {
    'best_epoch': training.get('best_epoch'),
    'completed_epochs': training.get('completed_epochs'),
    'early_stopping': training.get('early_stopping'),
    'overfit_monitor': training.get('overfit_monitor'),
    'baseline_comparison': training.get('baseline_comparison'),
    'analysis_split': analysis.get('analysis_split'),
    'top1_accuracy': analysis.get('top1_accuracy'),
    'macro_f1': analysis.get('macro_f1'),
    'generalization': analysis.get('generalization'),
    'class_support': analysis.get('class_support'),
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('\n5 epoch cuối:')
for item in training.get('history', [])[-5:]:
    print({key: item.get(key) for key in ('epoch', 'train_loss', 'validation_loss', 'train_top1', 'validation_top1', 'loss_generalization_gap', 'top1_generalization_gap', 'early_stopping_bad_epochs', 'overfit_signal', 'overfit_bad_epochs')})
for name in ('training_curves.png', 'selected_50_official_split_counts.png', f'{ANALYSIS_SPLIT}_confusion_matrices.png', f'{ANALYSIS_SPLIT}_per_class_metrics.png', f'{ANALYSIS_SPLIT}_top_confusions.png', f'{ANALYSIS_SPLIT}_confidence_histogram.png'):
    path = DUAL_ROOT / name
    if path.is_file():
        print('\n', name); display(Image(filename=str(path)))

## Diễn giải đúng kết quả

- `early_stopping.reason=validation_loss_no_improvement`: validation loss không cải thiện đủ lâu.
- `early_stopping.reason=sustained_overfit_signal`: loss gap và top-1 gap lớn, đồng thời validation loss đã tụt khỏi best liên tiếp đủ số epoch.
- Kết quả đại diện luôn lấy từ `best_checkpoint.pt`, không lấy epoch cuối.
- Macro-F1 là trung bình F1 của 50 từ; từ ít clip và nhiều clip có trọng số ngang nhau.
- Nếu phân tích test, notebook 09 phải được chạy lại với `RUN_TEST=True` sau khi đã chốt cấu hình. Không dùng kết quả test để sửa tiếp mô hình.